# MEER Free Factory — Colab Notebook

**Goal:** one free end-to-end chain on Google Colab's FREE T4 GPU — story line → still (free image model) → animated clip (free video model) → ffmpeg staging → GitHub upload. No PC, no card, no paid credits.

**How to run:**
1. Runtime → Change runtime type → Hardware accelerator: **GPU** (free tier, expect Tesla T4 16 GB).
2. Set the GitHub secret first: left sidebar → Secrets (🔑) → add **`GITHUB_TOKEN`** = a classic token with **`repo`** scope. The notebook reads it with `userdata.get()` and **never prints it**. No token is hardcoded anywhere.
3. Runtime → Run all. Expected wall time on a free T4: ~5 min setup+downloads (first run only, models ~18 GB total) + ~1 min still + ~10 min clip (CogVideoX-5B-I2V, official Colab figure; INT8 ~30 min only if OOM) + ~1 min staging/upload. If the clip is too slow, the fallbacks are LTX-Video 2B distilled (est. 1–3 min/clip, lower fidelity) or Wan 2.2 TI2V-5B (est. 5–10 min/clip, no official notebook) — see the fallback note in the config cell.

**Honest limits (read before relying on this):** free Colab sessions last at most 12 h, idle timeout ~90 min, and a T4 is **not guaranteed** (you may queue or get a weaker GPU at peak hours). One 6-second clip ≈ 30 min on this config, so one session yields only a handful of clips. This is **free forever, bounded throughput** — not unlimited. See the morning report for the full numbers and the Kaggle/Modal second doors.

In [ ]:
# ---- CONFIG: edit these for a real story run ----
REPO = "Hamzaagsm/videomaker"      # GitHub repo (owner/name)
BRANCH = "main"
HERO = "Bella"                       # Rayo / Bella / Rusty / Bruno
LINE = "Good morning, friends! I am Bella - and I have a big idea!"  # story line for this clip
STORY_HASH = "test-colab-factory"     # real runs: the 16-hex FNV-1a hash of the full script
CLIP_NO = 1                          # scene number within the story

# Test upload paths (safe: nothing in the app reads test/). Real runs:
#   clip -> f"vault/{STORY_HASH}/..." is the MANIFEST path; clips go to
#   f"suspense-clips/susp-vault-{STORY_HASH}-{CLIP_NO}.mp4"
#   (journey stories use adv-clips/susp-jrn-<id>-<n>.mp4 instead).
REPO_PATH_STILL = f"test/colab-factory/bella-colab-factory-still-{CLIP_NO}.png"
REPO_PATH_CLIP = f"test/colab-factory/bella-colab-factory-clip-{CLIP_NO}.mp4"

# Models (defaults = the researched ones; see fallback note below)
STILL_MODEL = "playgroundai/playground-v2.5-1024px-aesthetic"  # Playground v2.5, SDXL arch, ~8 GB, best cartoon colour
# Fallback still model: "stabilityai/stable-diffusion-xl-base-1.0" (SDXL base, same VRAM).
VIDEO_MODEL = "THUDM/CogVideoX-5b-I2V"                # image-to-video, character stays the same
# Official free-T4 Colab notebook for this exact model:
# https://colab.research.google.com/drive/17CqYCqSwz39nZAX2YyonDxosVKUZGzcX?usp=sharing
# FALLBACKS if 5B-I2V is too slow on your T4 (do NOT use CogVideoX-2B for I2V —
# no official 2B-I2V checkpoint exists; and do NOT use CogVideoX1.5-5B-I2V —
# it needs 60-90+ min/clip on a T4):
#   (a) LTX-Video 2B distilled — 8 steps, native 24 fps, est. 1-3 min/clip,
#       lower fidelity; (b) Wan 2.2 TI2V-5B — native 720p/24 fps I2V,
#       est. 5-10 min/clip, needs a self-built pipeline (no official notebook).
NUM_FRAMES = 49        # 49 frames @ 8 fps = ~6 seconds
NUM_STEPS = 50
GUIDANCE = 6
SEED = 42

STILL_PROMPT = ("Bella, a tall cute YELLOW cartoon car with a happy smiling face on the front, "
                "big friendly headlights like kind eyes, cheerful cartoon style for kids, shiny yellow paint, "
                "at the school gate with excited children cheering and waving around her, sunny schoolyard, "
                "vertical portrait 9:16")
VIDEO_PROMPT = ("Bella, a tall cute yellow cartoon car with a happy smiling face and big friendly "
                "headlights like kind eyes, driving forward happily, wheels turning, background moving past, "
                "sunny day, cheerful cartoon style for kids")

print("Config OK:", HERO, "| line:", LINE[:60], "| clip:", REPO_PATH_CLIP)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# PINNED (2026-10-25 06:00): unpinned --upgrade pulled transformers 5.20.2, which
# broke the LTX notebook's T5 load (device_map="disk" removed). Pin to the versions
# the LTX quality-pass run proved working on this Colab env. Torch untouched.
!pip install -q diffusers==0.42.0 transformers==5.20.2 accelerate==1.15.0 imageio-ffmpeg requests

## 1. Still — free image model (Playground v2.5, SDXL architecture) on the same T4

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

still_pipe = StableDiffusionXLPipeline.from_pretrained(STILL_MODEL, torch_dtype=torch.bfloat16)
still_pipe.enable_model_cpu_offload()
still_pipe.vae.enable_tiling()

image = still_pipe(
    prompt=STILL_PROMPT,
    negative_prompt="blurry, distorted, scary, dark, extra face, deformed, low quality",
    width=768, height=1344,          # 9:16 portrait
    num_inference_steps=30,
    guidance_scale=7.0,
    generator=torch.Generator(device="cuda").manual_seed(SEED),
).images[0]
image.save("still.png")
print("Still saved: still.png", image.size)

# Free the VRAM before the (much bigger) video model loads.
del still_pipe
torch.cuda.empty_cache()
print("VRAM freed for the video model")

## 2. Animated clip — CogVideoX image-to-video (the still becomes the first frame, so the character stays the same)

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

pipe = CogVideoXImageToVideoPipeline.from_pretrained(VIDEO_MODEL, torch_dtype=torch.bfloat16)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print("Video pipeline loaded OK")

image = load_image(image="still.png")
video = pipe(
    prompt=VIDEO_PROMPT,
    image=image,
    num_videos_per_prompt=1,
    num_inference_steps=NUM_STEPS,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE,
    generator=torch.Generator(device="cuda").manual_seed(SEED),
).frames[0]
export_to_video(video, "clip-raw.mp4", fps=8)
import os
print("Raw clip saved: clip-raw.mp4", os.path.getsize("clip-raw.mp4"), "bytes")

## 3. Staging — ffmpeg to the MEER contract (1080x1920, 24 fps, -bf 0, CRF 22, no audio)

In [ ]:
!ffmpeg -y -i clip-raw.mp4 -vf "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,fps=24" -an -c:v libx264 -bf 0 -g 48 -crf 22 clip-staged.mp4
!ffprobe -v quiet -print_format json -show_streams -show_format clip-staged.mp4 | python3 -c "import json,sys; d=json.load(sys.stdin); s=[x for x in d['streams'] if x['codec_type']=='video'][0]; print('staged:', s['width'],'x',s['height'],'fps',s['r_frame_rate'],'codec',s['codec_name'],'duration',d['format']['duration'],'size',d['format']['size'])"

## 4. Upload to GitHub via the API (token from Colab Secrets — never hardcoded, never printed)

In [ ]:
import base64, json, os, urllib.request, urllib.error
from google.colab import userdata

TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "Colab secret GITHUB_TOKEN is missing — add it under Secrets (key icon, left sidebar)"

def gh_upload(repo_path, local_path, message):
    with open(local_path, "rb") as f:
        content_b64 = base64.b64encode(f.read()).decode()
    base = f"https://api.github.com/repos/{REPO}/contents/{repo_path}"
    headers = {"Accept": "application/vnd.github+json", "Authorization": f"Bearer {TOKEN}",
               "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "meer-free-factory/1.0"}
    # Existing file? (need its sha to replace it)
    sha = None
    req = urllib.request.Request(base + f"?ref={BRANCH}", headers=headers, method="GET")
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            sha = json.loads(r.read().decode()).get("sha")
    except urllib.error.HTTPError as e:
        assert e.code == 404, f"GET {repo_path} failed: {e.code}"
    payload = {"message": message, "content": content_b64, "branch": BRANCH}
    if sha:
        payload["sha"] = sha
    req = urllib.request.Request(base, data=json.dumps(payload).encode(),
                                 headers={**headers, "Content-Type": "application/json"}, method="PUT")
    with urllib.request.urlopen(req, timeout=300) as r:
        data = json.loads(r.read().decode())
    assert r.status in (200, 201), f"PUT {repo_path} failed: {r.status}"
    print(f"Uploaded {repo_path} -> sha {data['content']['sha'][:10]} size {data['content']['size']}")
    return data["content"]["sha"]

gh_upload(REPO_PATH_STILL, "still.png", f"MEER free factory: test still {CLIP_NO} ({HERO})")
gh_upload(REPO_PATH_CLIP, "clip-staged.mp4", f"MEER free factory: test clip {CLIP_NO} ({HERO})")
print("DONE — still + staged clip are live on GitHub:")
print("  https://github.com/" + REPO + "/blob/" + BRANCH + "/" + REPO_PATH_CLIP)

In [ ]:
!ls -la still.png clip-raw.mp4 clip-staged.mp4